In [1]:
import sys
from pathlib import Path

print("Python:", sys.version.split()[0])
print("Folder:", Path.cwd().name)

for name in ["numpy", "pandas", "sklearn"]:
    try:
        __import__(name)
        print(name, "- ok")
    except ImportError:
        print(name, "- missing")

Python: 3.13.15
Folder: content
numpy - ok
pandas - ok
sklearn - ok


In [2]:
import csv
from pathlib import Path
import numpy as np

SEED = 42
N_ROWS = 600

DATA = Path("data") / "delivery_times.csv"


def make_delivery_csv(path=DATA):
    rng = np.random.default_rng(SEED)

    distance_km = np.round(
        rng.uniform(0.5, 12.0, N_ROWS), 2
    )

    prep_time_min = np.round(
        rng.uniform(5, 30, N_ROWS), 0
    )

    traffic_level = rng.integers(1, 4, N_ROWS)

    rain = rng.binomial(1, 0.25, N_ROWS)

    delivery_min = np.round(
        6.0
        + 3.1 * distance_km
        + 0.65 * prep_time_min
        + 4.2 * traffic_level
        + 5.5 * rain
        + rng.normal(0, 2.5, N_ROWS),
        1
    )

    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)

        w.writerow([
            "distance_km",
            "prep_time_min",
            "traffic_level",
            "rain",
            "delivery_min"
        ])

        for i in range(N_ROWS):
            w.writerow([
                distance_km[i],
                int(prep_time_min[i]),
                int(traffic_level[i]),
                int(rain[i]),
                delivery_min[i]
            ])

    return path


if not DATA.exists():
    make_delivery_csv()

print("dataset ready:", DATA)

dataset ready: data/delivery_times.csv


In [3]:
import pandas as pd

orders = pd.read_csv(DATA)

print(orders.shape)
print(orders.head())

(600, 5)
   distance_km  prep_time_min  traffic_level  rain  delivery_min
0         9.40             17              1     0          51.3
1         5.55             24              2     1          54.2
2        10.37             28              3     0          67.7
3         8.52             23              2     0          51.2
4         1.58             29              2     1          42.1


In [4]:
FEATURES = [
    "distance_km",
    "prep_time_min",
    "traffic_level",
    "rain"
]

TARGET = "delivery_min"

X = orders[FEATURES]
y = orders[TARGET]

print(X.head(3))
print(y.head(3))

   distance_km  prep_time_min  traffic_level  rain
0         9.40             17              1     0
1         5.55             24              2     1
2        10.37             28              3     0
0    51.3
1    54.2
2    67.7
Name: delivery_min, dtype: float64


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(len(X_train), len(X_test))

480 120


In [6]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

average_time = y_train.mean()

baseline_guesses = np.full(
    len(y_test),
    average_time
)

baseline_mae = mean_absolute_error(
    y_test,
    baseline_guesses
)

print("baseline MAE:", round(baseline_mae, 2))

baseline MAE: 10.32


In [7]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train, y_train)

print("trained on", len(X_train), "orders")

trained on 480 orders


In [8]:
predictions = model.predict(X_test)

mae = mean_absolute_error(
    y_test,
    predictions
)

rmse = float(
    np.sqrt(
        mean_squared_error(
            y_test,
            predictions
        )
    )
)

print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))

MAE: 1.92
RMSE: 2.48


In [9]:
learned = pd.DataFrame({
    "feature": FEATURES,
    "minutes per unit": model.coef_.round(2),
})

print(learned.to_string(index=False))

print("intercept:", round(model.intercept_, 1))

      feature  minutes per unit
  distance_km              3.07
prep_time_min              0.65
traffic_level              4.13
         rain              5.55
intercept: 6.4


In [10]:
new_order = pd.DataFrame([{
    "distance_km": 5.0,
    "prep_time_min": 20,
    "traffic_level": 2,
    "rain": 0,
}])

prediction = model.predict(new_order)[0]

print(round(prediction, 1))

43.0


In [11]:
#T1
median_time = y_train.median()

median_guesses = np.full(
    len(y_test),
    median_time
)

median_mae = mean_absolute_error(
    y_test,
    median_guesses
)

print("Mean baseline MAE:", round(baseline_mae, 2))
print("Median baseline MAE:", round(median_mae, 2))

if median_mae < baseline_mae:
    print("Median baseline is better.")
else:
    print("Mean baseline is better.")

Mean baseline MAE: 10.32
Median baseline MAE: 10.33
Mean baseline is better.


In [14]:
#T2
X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=7
)

model_2 = LinearRegression()

model_2.fit(X_train_2, y_train_2)

predictions_2 = model_2.predict(X_test_2)

mae_2 = mean_absolute_error(
    y_test_2,
    predictions_2
)

print("Training rows:", len(X_train_2))
print("Testing rows:", len(X_test_2))
print("Test MAE:", round(mae_2, 2))

Training rows: 420
Testing rows: 180
Test MAE: 2.1


In [15]:
#T3
def predict_delivery(distance_km, prep_time_min, traffic_level, rain):
    new_order = pd.DataFrame([{
        "distance_km": distance_km,
        "prep_time_min": prep_time_min,
        "traffic_level": traffic_level,
        "rain": rain
    }])

    prediction = model.predict(new_order)[0]

    return round(prediction, 1)


result = predict_delivery(
    3,
    15,
    1,
    1
)

print(result)

35.0
